In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.analytics.yield_analytics import YieldAnalytics
from src.utils.config import Config

print("✅ Ready")

## 1. Load Data and Initialize

In [ ]:
config = Config()
df = pd.read_parquet(config.data_dir / 'staging' / 'test_results.parquet')

# Initialize yield analytics
ya = YieldAnalytics(df)

print(f"Loaded {len(df):,} test records")
print(f"Devices: {df['device_id'].nunique()}")
print(f"Tests: {df['test_name'].nunique()}")

## 2. Overall Yield Metrics

In [ ]:
# Get comprehensive yield summary
summary = ya.generate_summary()

print("="*60)
print("YIELD SUMMARY")
print("="*60)
print(f"\n📊 Device-Level:")
print(f"   Overall Yield: {summary['overall_yield']:.2f}%")
print(f"   Passing Devices: {summary['passing_devices']:,} / {summary['total_devices']:,}")

print(f"\n🧪 Test-Level:")
print(f"   Test Yield: {summary['test_yield']:.2f}%")
print(f"   Passing Tests: {summary['passing_tests']:,} / {summary['total_tests']:,}")

print(f"\n⚠️  Lowest Yielding Test:")
print(f"   {summary['lowest_yield_test']}: {summary['lowest_yield_value']:.2f}%")

## 3. Yield by Lot/Wafer

In [ ]:
# Lot-level yield
lot_yield = ya.yield_by_lot()
print("\nYield by Lot:")
print(lot_yield)

# Visualize
fig = ya.plot_yield_trend(by='lot')
fig.show()

In [ ]:
# Wafer-level yield (if available)
wafer_yield = ya.yield_by_wafer()
if not wafer_yield.empty:
    print("\nYield by Wafer:")
    print(wafer_yield)
    
    fig = ya.plot_yield_trend(by='wafer')
    fig.show()

## 4. Test-Level Yield Analysis

In [ ]:
# Yield by individual test
test_yield = ya.yield_by_test()
print("\nYield by Test:")
print(test_yield.to_string())

# Visualize
fig = px.bar(
    test_yield,
    x='test_name',
    y='yield',
    title='Yield by Test',
    labels={'yield': 'Yield (%)', 'test_name': 'Test'},
    text='yield'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.add_hline(y=90, line_dash="dash", line_color="red")
fig.update_layout(xaxis_tickangle=45)
fig.show()

## 5. Pareto Analysis of Failures

In [ ]:
# Pareto chart for top failing tests
pareto_df = ya.failing_tests_pareto(top_n=10)

print("\nTop Failing Tests (Pareto):")
print(pareto_df.to_string(index=False))

fig = ya.plot_pareto(top_n=10)
fig.show()

# 80/20 analysis
tests_for_80pct = (pareto_df['cumulative_pct'] <= 80).sum()
print(f"\n📈 Pareto Insight:")
print(f"   {tests_for_80pct} tests account for 80% of all failures")
print(f"   Focus on these for maximum yield improvement")

## 6. Yield Distribution Analysis

In [ ]:
# Device yield distribution
fig = ya.plot_yield_distribution()
fig.show()

# Calculate yield buckets
device_pass_rate = df.groupby('device_id')['result'].apply(
    lambda x: (x == 'pass').sum() / len(x) * 100
)

print("\nDevice Yield Distribution:")
print(f"   100% yield: {(device_pass_rate == 100).sum()} devices")
print(f"   90-99%: {((device_pass_rate >= 90) & (device_pass_rate < 100)).sum()} devices")
print(f"   80-89%: {((device_pass_rate >= 80) & (device_pass_rate < 90)).sum()} devices")
print(f"   <80%: {(device_pass_rate < 80).sum()} devices")

## 7. Bin Analysis

In [ ]:
# Bin distribution
bin_summary = df.groupby('bin').agg({
    'device_id': 'nunique',
    'result': lambda x: (x == 'pass').sum() / len(x) * 100
}).rename(columns={'device_id': 'device_count', 'result': 'bin_yield'})

print("\nBin Summary:")
print(bin_summary)

fig = px.sunburst(
    df.groupby(['bin', 'result']).size().reset_index(name='count'),
    path=['bin', 'result'],
    values='count',
    title='Bin Distribution (Hierarchical)',
    color='result',
    color_discrete_map={'pass': 'green', 'fail': 'red'}
)
fig.show()

## 8. Actionable Insights

In [ ]:
# Generate action items based on analysis
print("="*60)
print("ACTIONABLE INSIGHTS & RECOMMENDATIONS")
print("="*60)

# 1. Low yield tests
low_yield_tests = test_yield[test_yield['yield'] < 90]['test_name'].tolist()
if low_yield_tests:
    print(f"\n🎯 Priority 1: Improve Low-Yield Tests")
    for test in low_yield_tests:
        yield_val = test_yield[test_yield['test_name'] == test]['yield'].values[0]
        print(f"   - {test}: {yield_val:.1f}% (Target: 90%)")
    print(f"   Action: Root cause analysis for these {len(low_yield_tests)} tests")

# 2. Pareto failures
top_3_failures = pareto_df.head(3)
print(f"\n🔍 Priority 2: Focus on Top Failure Drivers")
for _, row in top_3_failures.iterrows():
    print(f"   - {row['test_name']}: {row['failures']} failures ({row['percentage']:.1f}%)")
print(f"   Action: These 3 tests drive {top_3_failures['percentage'].sum():.1f}% of failures")

# 3. Overall assessment
overall = summary['overall_yield']
print(f"\n📊 Overall Assessment:")
if overall >= 95:
    print(f"   ✅ Excellent: {overall:.2f}% yield")
elif overall >= 90:
    print(f"   ⚠️  Good: {overall:.2f}% yield, room for improvement")
else:
    print(f"   ❌ Needs Attention: {overall:.2f}% yield below target")

print(f"\n💡 Next Steps:")
print(f"   1. Deep dive into top 3 failing tests")
print(f"   2. Analyze correlation between test failures")
print(f"   3. Investigate bin patterns")
print(f"   4. Review test limits and specifications")

## 9. Summary

**What we learned:**
- ✅ Comprehensive yield analysis (device, test, lot, wafer)
- ✅ Pareto analysis for failure prioritization
- ✅ Yield distribution patterns
- ✅ Bin analysis
- ✅ Actionable insight generation

**Key Metrics:**
- Overall Yield: **{:.2f}%**
- Tests below target: **{}**
- Top failure driver accounts for **{:.1f}%** of failures

**Next Steps:**
- Statistical analysis (hypothesis testing)
- Machine learning for yield prediction
- Automated dashboard deployment